# Santa's gift cards, one kid at a time

Run this notebook on a **GPU**: a DevLab (JupyterLab on a GPU), with the `course (GPU)` kernel. It makes one kid's card step by step, so you can look at, listen to and change every result before you run the whole batch as a Job.

The functions come from [batch_inference.py](batch_inference.py), and the Job runs the same file. What works here is what the Job runs.

    kid profile -> text -> illustration -> card -> clip -> greeting -> mix

GPU: step 4 (the clip) runs a video model that needs an H100. On an L40S you can still run steps 1, 2, 3 and 5: skip the cells of steps 4 and 6 and the batch at the end.

> The GPU bills while the DevLab runs, including while you read. Delete the DevLab when you're done.

## Setup

Once per DevLab, in a **Terminal** (not in a cell). These are the packages the clip, greeting and mix steps need on top of the course GPU environment. `[verify on first run]`

```bash
uv pip install "kokoro>=0.9.2" ftfy imageio imageio-ffmpeg
```

In [ ]:
import sys
from pathlib import Path

import torch
from IPython.display import Audio, Video, display

CODE = next(p for p in (Path.cwd(), Path.cwd() / "code") if (p / "batch_inference.py").is_file())
sys.path.insert(0, str(CODE))

from batch_inference import (  # noqa: E402
    animate,
    card_seconds,
    draw_illustration,
    free_gpu,
    load_image_model,
    load_kids,
    load_video_model,
    load_voice,
    mix,
    pick_music,
    render_card,
    run,
    speak,
    write_text,
)
from models import SANA, WAN_I2V  # noqa: E402

## A kid

One row of `kids.csv` becomes a `KidProfile`: a typed record. The row already has the wish, the greeting and the illustration prompt. A row with a missing name or an age outside 1–18 fails here, before any GPU work. Change the index to pick another kid.

In [ ]:
kid = load_kids(1)[0]
OUT = Path("runs/santa-dev/cards") / kid.id  # the same layout the Job writes: cards/<id>/
OUT.mkdir(parents=True, exist_ok=True)
kid

## 1. Text

The kid's row in, `card.json` out. `write_text` copies the wish, the greeting and the illustration prompt from the row. Every later step reads that file.

In [ ]:
card = write_text(kid)
(OUT / "card.json").write_text(card.model_dump_json(indent=2))
print(card.model_dump_json(indent=2))

## 2. Illustration (Sana)

First the raw call: load Sana once, then call the pipeline with the prompt, the lesson's settings (`SANA.params`: 512×512, 20 steps) and a fixed seed.

In [ ]:
sana = load_image_model()
generator = torch.Generator(device="cuda").manual_seed(SANA.seed)
illustration = sana(prompt=card.image_prompt, **SANA.params, generator=generator).images[0]
display(illustration)

`draw_illustration()` is the same call as a function; the Job uses it. Same prompt, settings and seed, so you get the same picture.

In [ ]:
illustration = draw_illustration(sana, card.image_prompt)
illustration.save(OUT / "illustration.png")
display(illustration)

## 3. Card (Pillow, no GPU)

The illustration, the kid's name and the wish on one 1024×1400 card.

In [ ]:
card_image = render_card(illustration, card)
card_image.save(OUT / "card.png")
display(card_image.resize((512, 700)))

Sana is still in GPU memory. Free it before the video model loads; the batch run does the same between steps.

In [ ]:
del sana
free_gpu()

## 4. Clip (Wan2.2 image-to-video, H100)

Animates the illustration: a slow camera drift, falling snow, twinkling lights. The settings come from `WAN_I2V`. The first run downloads the model's weights, which takes a while.

In [ ]:
print(WAN_I2V.params, f"{WAN_I2V.fps} fps, {WAN_I2V.duration_s:.1f} s")
wan = load_video_model()
animate(wan, OUT / "illustration.png", OUT / "clip.mp4")
display(Video(str(OUT / "clip.mp4"), embed=True))

In [ ]:
del wan
free_gpu()

## 5. Greeting (Kokoro text-to-speech)

In [ ]:
voice = load_voice()
seconds = speak(voice, card.greeting, OUT / "greeting.wav")
print(card.greeting, f"({seconds:.1f} s)")
display(Audio(str(OUT / "greeting.wav")))

In [ ]:
del voice
free_gpu()

## 6. Mix (ffmpeg, no GPU)

Clip + greeting + quiet background music in one MP4. `card_seconds()` decides the length: the clip's, or the greeting's plus half a second if that's longer.

In [ ]:
greeting = OUT / "greeting.wav"
mix(OUT / "clip.mp4", greeting, pick_music(card.mood), OUT / "card.mp4", card_seconds(greeting))
display(Video(str(OUT / "card.mp4"), embed=True))

## The batch: the same steps for three kids

`run()` goes step by step, not kid by kid: it loads one model, makes that step's file for every kid, saves it, frees the GPU and moves on. Watch the log: each model loads once, and every finished file gets a line.

In [ ]:
cards = run(n=3, out=Path("runs/santa-dev-batch"))
sorted(str(p.relative_to(cards)) for p in cards.rglob("*") if p.is_file())

## Next

Run the same file as a Job that writes to your bucket: see "Run the Job" in [lesson 04](../04-serving-vs-batch.md). Then delete this DevLab.